# 1.3 模型到底在猜什麼？


如果把「貓看狗。」完整交給程式，又要求它寫出同一句，它只要照抄就能成功。我們要練的則是：還沒看到後面的字，先猜下一字。這個差別必須寫進資料，否則後面再漂亮的計算都會在學錯任務。延續 [1.1](https://birdhackor.github.io/tiny-perceptron-vlm/1.1.html) 的文字記錄，本節把一句話拆成一串「問題、答案」，不需要先懂模型如何學習。

拿「貓看狗。」逐位置看：目前是「貓」，下一字答案為「看」；目前是「看」，答案為「狗」；目前是「狗」，答案為「。」。句號之後沒有提供下一字，所以暫時不拿它當問題。這裡每個問題只看一字，後續章節才會擴大前文；但答案永遠是問題位置右邊一格的字。

把原句叫 `s`，取掉最後一項得到輸入 `x`，取掉第一項得到答案 `y`。Python 的切片（slice）可一次取出一段清單：`s[:-1]` 從開頭取到最後一項之前，`s[1:]` 從位置 1 取到結尾。負索引 -1 代表最後一項，切片的右端不包含在結果中。索引與清單背景見 [W.2](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.2)。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
s = list("貓看狗。")
x = s[:-1]
y = s[1:]
print(x)
print(y)
for current, answer in zip(x, y, strict=True):
    print("看到", current, "下一字", answer)
assert len(x) == len(y) == 3

`list` 把字串拆成依序排列的字，原句 `s` 共有四個項目；問題 `x` 和答案 `y` 則各三個。前兩行輸出分別是 `['貓', '看', '狗']` 和 `['看', '狗', '。']`，長度都是三。`zip` 把兩份清單同一位置配對，迴圈逐對印出三道問題；`strict=True` 要求兩份長度相同，不同就報錯，免得悄悄少掉答案。最後檢查同時確認兩邊都是三項。

下圖上列藍框是問題 x，下列綠框是同位置的答案 y。每個向下箭頭代表一題，不表示把同一個字複製下來：左邊是貓配看，中間看配狗，右邊狗配句號。最上方的原句有四字，問題與答案則各三格。


In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/foundations_shift.svg")))

這個右移一格的對齊叫 shift。名字雖像「移動」，實際並沒有修改原句，而是用兩個切片建立對應關係。後面用 ID 取代字時，也要保持同樣的對齊。如果輸入與答案都用 `s[:-1]`，第一題就變成「看到貓，回答貓」，整份資料會獎勵複製目前字，而不是預測下一字。

也別把 shift 和「防止偷看」混成一件事。shift 決定哪個位置該配哪個答案；當模型一次處理多個位置時，還需要限制每個位置只能讀到當下及以前的資料。後者會在 [3.6](https://birdhackor.github.io/tiny-perceptron-vlm/3.6.html) 說明。現在用單字問題，可以直接看出答案沒有塞在輸入裡。

練習先在紙上寫下「鳥飛。」的所有問題答案，再把 `s` 的字串改成它，並把檢查中的 3 改成 2。核對應有「鳥→飛、飛→。」兩對。最後故意把 `y` 改成 `s[:-1]`，觀察輸出如何變成自己配自己；這次長度檢查仍會通過，提醒我們形式對齊不代表任務正確，還要逐題讀一遍。
